In [4]:
import json
import pandas as pd

In [5]:
df = pd.read_json('alerts_combined.json')

In [69]:
df[df["channel"] == 'monitoring-ops-cx'].error_type.isna().sum()

np.int64(0)

In [70]:
df

,ts,timestamp,source,priority,service,condition,threshold,policy,incidents,channel,error_type,error_message
0,1.749647e+09,2025-06-11 07:09:39-06:00,New Relic,high,Princess,High Application Response Time gral,>1800ms/5min,Golden Signals,1.0,sre,NaN,NaN
1,1.749643e+09,2025-06-11 05:55:05-06:00,New Relic,critical,Cerberus,High Application Error percentage,baseline/10min,Golden Signals,1.0,sre,NaN,NaN
2,1.749643e+09,2025-06-11 05:54:15-06:00,New Relic,critical,tesseract,Low Application Throughput,baseline/10min,Golden Signals,1.0,sre,NaN,NaN
3,1.749639e+09,2025-06-11 04:43:34-06:00,New Relic,critical,Cerberus,Low Application Throughput,baseline/10min,Golden Signals,1.0,sre,NaN,NaN
4,1.749631e+09,2025-06-11 02:36:38-06:00,New Relic,critical,data-team,RDS CPU Usage gral,>60%/10min,Golden Signals,2.0,sre,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
453,1.742001e+09,2026-03-27 15:46:23-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.0,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...
454,1.742001e+09,2026-03-27 15:57:45-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.0,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...
455,1.742001e+09,2026-03-27 16:06:33-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.0,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...
456,1.742000e+09,2026-03-27 16:12:02-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.0,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...


In [53]:
conteo = df.error_message.value_counts()
tabla = pd.DataFrame({
    "frecuencia": conteo,
    "porcentaje": conteo / conteo.sum() * 100,
    "acumulado":  conteo.cumsum() / conteo.sum() * 100
})
print(tabla)

                                                    frecuencia  porcentaje  \
error_message                                                                
INSUFFICIENT_FUNDS                                          27   28.421053   
IMPOSSIBLE_TO_CHARGE                                        25   26.315789   
Tarjeta declinada. Intenta con otra tarjeta o m...          22   23.157895   
El banco emisor rechazó el pago sin más detalle...          12   12.631579   
INSTRUMENT_DECLINED                                          4    4.210526   
Fondos insuficientes                                         1    1.052632   
BLOCKED_CREDIT_CARD                                          1    1.052632   
PAYPAL_UNAVAILABLE                                           1    1.052632   
Error from Mercadopago                                       1    1.052632   
Card declined. Please try with another card or ...           1    1.052632   

                                                     acumulado 

In [80]:
pd.set_option("display.float_format", "{:.2f}".format)


In [84]:
df["diff_dias"] = df["diff_dias"].round(0)


In [71]:
df["fecha"] = pd.to_datetime(df["ts"], unit="s")  # o "ms"
print(df["fecha"])
# 2024-06-11 18:30:56


0     2025-06-11 13:09:39.517959118
1     2025-06-11 11:55:05.830949068
2     2025-06-11 11:54:15.028158903
3     2025-06-11 10:43:34.938539028
4     2025-06-11 08:36:38.335369110
                   ...             
453   2025-03-15 01:13:20.000000000
454   2025-03-15 01:08:20.000000000
455   2025-03-15 01:03:20.000000000
456   2025-03-15 00:58:20.000000000
457   2025-03-15 00:53:20.000000000
Name: fecha, Length: 458, dtype: datetime64[ns]


In [74]:
df["fecha"] = (
    pd.to_datetime(df["ts"], unit="s")
    .dt.tz_localize("UTC")
    .dt.tz_convert("America/Mexico_City")
)



In [78]:
# Asegúrate que ambas columnas sean datetime
df["fecha"]     = pd.to_datetime(df["fecha"])
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Calcular diferencia en segundos por fila
df["diff_dias"] = (df["fecha"] - df["timestamp"]).dt.total_seconds()/86400

# Máxima diferencia
print(df["diff_dias"].max())


0.7083333333333334


In [93]:
df[df['channel'] == 'monitoring-ops-cx'].diff_dias.value_counts()

diff_dias
-375.00    46
-378.00    23
-374.00    12
-376.00     9
-377.00     5
Name: count, dtype: int64

In [ ]:
1. Veo que casi todo es source = New Relic (455 de 458) y sólo hay 3 de Paypal Status
2. No hay diferencia porcentual en priority 51 % high 49% critical sin mayor distincion
3. 13 services de un total de 28, cubren el 90% en las alertas
4. 11 conditions de 23, representan el 90% en las alertas
5. 6 threshold de 28, representan el 70% en las alertas
6. No se realmente que es policy pero solo hay 4 categorias
7. El 94% de las incidents es menor igual a 2
8. Los channels son sre (79%) y monitoring-ops-cx (21%)
9. error_type y error_message tienen 79% de nulos, el resto de columnas o keys tienen menos de 1% o 0% de nulos
10. INSUFFICIENT_FUNDS, IMPOSSIBLE_TO_CHARGE, CARD_DECLINED, BANK_REJECTED son el 92% de los error_type de un total de 95 valores con esta columna
11. Al parecer 95 alertas el 100% de ellas son relacionadas a errores de pagos 
12. Parece ser que error_message tambien describe en su totalidad un problema de pago, creo que estas dos columnas de error_ pueden unificarse a una sola 
13. Todas los registros que tienen error_type pertenecen a 1 solo channel que es monitoring-ops-cx
14. La diff en días entre ts y timestamp es de 0 o 1 para el channel sre y de 375 a 378 para el channel monitoring-ops-cx  

SyntaxError: invalid syntax (2328481756.py, line 1)

In [49]:
# Conteo absoluto
df.isnull().sum()

# Con porcentaje (más útil)
nulos = pd.DataFrame({
    "nulos":      df.isnull().sum(),
    "porcentaje": df.isnull().sum() / len(df) * 100
})

# Solo mostrar columnas que SÍ tienen nulos, ordenado de mayor a menor
nulos = nulos[nulos["nulos"] > 0].sort_values("porcentaje", ascending=False)
print(nulos)


               nulos  porcentaje
error_type       363   79.257642
error_message    363   79.257642
priority           3    0.655022
threshold          3    0.655022
policy             3    0.655022
incidents          3    0.655022


In [104]:
df[df['channel'] == 'monitoring-ops-cx'][50:95][['timestamp', 'fecha']]
# timestamp del 24 al 27 marzo 2026
# ts del 14 al 15 de marzo 2025

,timestamp,fecha
413,2026-03-24 16:47:43-06:00,2025-03-14 22:33:20-06:00
414,2026-03-24 16:52:00-06:00,2025-03-14 22:28:20-06:00
415,2026-03-24 16:56:13-06:00,2025-03-14 22:23:20-06:00
416,2026-03-24 23:16:47-06:00,2025-03-14 22:18:20-06:00
417,2026-03-24 23:22:55-06:00,2025-03-14 22:13:20-06:00
418,2026-03-24 23:31:13-06:00,2025-03-14 22:08:20-06:00
419,2026-03-24 23:40:17-06:00,2025-03-14 22:03:20-06:00
420,2026-03-25 00:10:18-06:00,2025-03-14 21:58:20-06:00
421,2026-03-25 14:55:34-06:00,2025-03-14 21:53:20-06:00
422,2026-03-25 19:26:32-06:00,2025-03-14 21:48:20-06:00


In [105]:
df

,ts,timestamp,source,priority,service,condition,threshold,policy,incidents,channel,error_type,error_message,fecha,diff_segundos,diff_dias
0,1749647379.52,2025-06-11 07:09:39-06:00,New Relic,high,Princess,High Application Response Time gral,>1800ms/5min,Golden Signals,1.00,sre,NaN,NaN,2025-06-11 07:09:39.517959118-06:00,0.52,0.00
1,1749642905.83,2025-06-11 05:55:05-06:00,New Relic,critical,Cerberus,High Application Error percentage,baseline/10min,Golden Signals,1.00,sre,NaN,NaN,2025-06-11 05:55:05.830949068-06:00,0.83,0.00
2,1749642855.03,2025-06-11 05:54:15-06:00,New Relic,critical,tesseract,Low Application Throughput,baseline/10min,Golden Signals,1.00,sre,NaN,NaN,2025-06-11 05:54:15.028158903-06:00,0.03,0.00
3,1749638614.94,2025-06-11 04:43:34-06:00,New Relic,critical,Cerberus,Low Application Throughput,baseline/10min,Golden Signals,1.00,sre,NaN,NaN,2025-06-11 04:43:34.938539028-06:00,0.94,0.00
4,1749630998.34,2025-06-11 02:36:38-06:00,New Relic,critical,data-team,RDS CPU Usage gral,>60%/10min,Golden Signals,2.00,sre,NaN,NaN,2025-06-11 02:36:38.335369110-06:00,0.34,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
453,1742001200.00,2026-03-27 15:46:23-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.00,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...,2025-03-14 19:13:20-06:00,-32646783.00,-378.00
454,1742000900.00,2026-03-27 15:57:45-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.00,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...,2025-03-14 19:08:20-06:00,-32647765.00,-378.00
455,1742000600.00,2026-03-27 16:06:33-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.00,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...,2025-03-14 19:03:20-06:00,-32648593.00,-378.00
456,1742000300.00,2026-03-27 16:12:02-06:00,New Relic,high,Hairs,Payments rejected hairs CX,>=5/5min,CX,1.00,monitoring-ops-cx,CARD_DECLINED,Tarjeta declinada. Intenta con otra tarjeta o ...,2025-03-14 18:58:20-06:00,-32649222.00,-378.00


In [118]:
df.incidents.value_counts()

incidents
1.00    298
2.00    131
3.00     14
4.00      6
6.00      4
7.00      2
Name: count, dtype: int64

In [127]:
df.pivot_table(
    values="ts",
    index="incidents",
    columns="priority",
    aggfunc="count",
    fill_value=0,
    margins=True,       # agrega fila/columna "All" con totales
    margins_name="Total"
)


priority,critical,high,Total
incidents,,,
1.00,64,234,298
2.00,131,0,131
3.00,14,0,14
4.00,6,0,6
6.00,4,0,4
7.00,2,0,2
Total,221,234,455


In [117]:
df.pivot_table(
    values="ts",
    index="condition",
    columns="policy",
    aggfunc="count",
    fill_value=0,
    margins=True,       # agrega fila/columna "All" con totales
    margins_name="Total"
)

policy,CX,Golden Signals,Parco2.0 strict,Up_Satatus_Parco,Total
condition,,,,,
Apdex score,0,11,0,0,11
Carts Throughput high,0,17,0,0,17
Cerberus Throughput High,0,11,0,0,11
Error percentage high,0,1,0,0,1
External Scan Alert,0,1,0,0,1
High Application Error percentage,0,44,0,0,44
High Application Response Time gral,0,11,0,0,11
High response time,0,1,0,0,1
Low Application Throughput,0,9,0,0,9


In [141]:
df[(df.channel == 'sre') & (df.service == 'Orchestrator') & (df.threshold == '>60/5min')][['fecha','threshold','service']][0:50].sort_values('fecha')

,fecha,threshold,service
361,2025-05-26 16:07:41-06:00,>60/5min,Orchestrator
284,2025-05-31 11:27:50-06:00,>60/5min,Orchestrator
280,2025-05-31 12:17:53-06:00,>60/5min,Orchestrator
279,2025-05-31 12:37:44-06:00,>60/5min,Orchestrator
276,2025-05-31 13:12:43-06:00,>60/5min,Orchestrator
274,2025-05-31 13:51:01-06:00,>60/5min,Orchestrator
263,2025-05-31 15:27:45-06:00,>60/5min,Orchestrator
257,2025-05-31 15:57:59-06:00,>60/5min,Orchestrator
253,2025-05-31 16:12:35-06:00,>60/5min,Orchestrator
229,2025-06-01 15:02:46-06:00,>60/5min,Orchestrator
